# Dynamics 08 — Programmes transparents comme institutions : CUPOD, DUPOC, PrudentBot, CIMCIC

**Navigation** : [<< Dynamics-07-Equilibres-par-Simulation-Python](Dynamics-07-Equilibres-par-Simulation-Python.ipynb) | [Index](README.md) | [06i-Capstone >>](../GameTheory-06i-Ensembles-Limites-Poincare-Bendixson-Python.ipynb) | [09-BiensPublics >>](Dynamics-09-Biens-Publics-N-Personnes-CSharp.ipynb)

Dans le jeu d'actions du Dilemme du prisonnier (PD), `D` domine strictement `C` et
`(D, D)` est l'unique équilibre de Nash, même si les joueurs connaissent les codes. Le jeu
de **programmes** modifie l'espace des stratégies : remplacer son programme peut changer
la réaction adverse au code publié. Critch, Dennis et Russell (2022) étudient deux
**institutions** — ou deux programmes — qui
lisent mutuellement leurs statuts publics (« bylaws », code source) avant de jouer une partie
unique. Ils y montrent que des institutions formelles qui devraient se trahir coopèrent, et
réciproquement (arXiv:2208.07006, p. 1 et 3).

Ce notebook en tire un modèle **borné et terminant** dont les quatre agents sont `CUPOD`,
`DUPOC`, `PrudentBot` et `CIMCIC`, exécutable pas à pas, sans prétendre trancher une question
ouverte.
**Source primaire du port** : [notebook Python versionné](Dynamics-08-Institutions-Transparentes-Python.ipynb). Bibliographie héritée ; les sept textes C# sont compilés puis exécutés avec Roslyn.


## 0. Source primaire et conventions de nommage

**Source primaire de ce travail** : le carnet Python versionné. La notice bibliographique héritée ci-dessous est conservée sans revendiquer une nouvelle lecture du texte : Andrew Critch, Michael Dennis, Stuart Russell, « Cooperative and
uncooperative institution designs: Surprises and problems in open-source game theory »,
arXiv:2208.07006 (v1, 15 août 2022), Center for Human-Compatible AI, UC Berkeley.

| Élément cité | Localisation exacte |
| --- | --- |
| Cadre « open-source game theory », lecture du code source avant l'action | p. 3 |
| Définition de `CUPOD` (coopérer sauf preuve de défection) | p. 9 |
| Définition de `DUPOC` (faire défection sauf preuve de coopération) | p. 9 et 10 |
| Exemples 2.1, 3.1 et 3.2 (rôle de la borne `k`) | p. 7, 10 et notes 3 et 4 |
| Propositions 3.1 et 3.2 (non-exploitabilité de `CUPOD` et de `DUPOC`) | p. 11 |
| Théorème 3.4 : pour `k` grand, `outcome(CUPOD(k), CUPOD(k)) == (D, D)` | p. 13 |
| Théorème 3.7 : pour `k` grand, `outcome(DUPOC(k), DUPOC(k)) == (C, C)` | p. 14 |
| Lemme 3.6 (PBLT, Parametric Bounded Löb Theorem) | p. 13 |
| `PDUPOC` et Théorème 4.1 | p. 19 |
| Définition de `CIMCIC` et Proposition 5.1 | p. 20 et 21 |
| Théorème 5.2 : (a) `CIMCIC` contre `CIMCIC`, (b) `DUPOC` contre `CIMCIC` | p. 21 et 22 |
| « Cooperative affidavit » pour des institutions de type DUPOC | p. 16 |
| `PrudentBot` (agent non borné de LaVictoire et al. 2017) et Open Problem 9 | p. 26 |
| Les dix problèmes ouverts | p. 15, 18, 20, 22, 24, 25, 26 et 27 |

**Conventions de nommage** : les analogues bornés des définitions publiées portent le nom publié
sans suffixe (`CUPOD`, `DUPOC`, `CIMCIC`). Les sigles anglais d'origine sont `CUPOD`
(`Cooperate Unless Proof Of Defection`) et `DUPOC` (`Defect Unless Proof Of Cooperation`, p. 9).
`PrudentBot_borne` est un **candidat** : l'existence
même d'une version bornée de PrudentBot est l'Open Problem 9 (p. 26), qui n'est pas résolu ici.
`CooperateBot`, `DefectBot` et `CooperateBotOpake` sont des agents de référence ou de contrôle ;
les deux premiers reprennent les noms de la p. 6, le troisième est un contrôle local.

## Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :

- énoncer pourquoi `(D, D)` est l'unique équilibre de Nash du PD canonique, et pourquoi la
  transparence des programmes change ce verdict (Critch-Dennis-Russell 2022, p. 3) ;
- définir une sémantique **bornée, totale et terminante** des agents `CUPOD`, `DUPOC`,
  `PrudentBot` et `CIMCIC`, à partir d'un certificat public et d'une clôture finie ;
- distinguer rigoureusement trois régimes de preuve : **observation finie**, **résultat
  reproduit**, **théorème cité** ;
- reproduire une matrice de confrontations avec **deux implémentations indépendantes** et
  comparer mécaniquement les relations et les matrices obtenues ;
- exhiber un **contrôle négatif** où la transparence seule ne crée pas la coopération ;
- lire le tableau des **dix problèmes ouverts** de l'article, dont l'Open Problem 3, laissé
  explicitement ouvert — ni exercice à solution attendue, ni résultat de notebook.

### Prérequis

- C# .NET et Microsoft.CodeAnalysis.CSharp.Scripting 4.14.0.
- La notion d'équilibre de Nash en stratégies pures, et le PD sous forme normale.
- Lecture utile (facultative) : `Dynamics-04-Open-Source-Game-Theory-Python.ipynb` pour le cadre
  des jeux-programmes, et `Dynamics-07-Equilibres-par-Simulation-Python.ipynb` pour un
  analogue fondé sur la simulation.

### Durée estimée : 45 minutes

## 1. Le Dilemme du prisonnier canonique

On pose la matrice canonique `(T=5, R=3, P=1, S=0)` avec `T > R > P > S` et `2R > T + S`.

| | C | D |
| --- | --- | --- |
| **C** | R, R | S, T |
| **D** | T, S | P, P |

`D` domine strictement `C` pour chaque joueur, quelle que soit l'action adverse : l'unique
équilibre de Nash en stratégies pures est `(D, D)`, quelle que soit la connaissance du code.
Dans le jeu de programmes étudié ensuite, une déviation change le code publié et peut
changer la réponse adverse : elle n'est donc pas une simple substitution d'action à réponse fixée.

In [1]:
#r "nuget: Microsoft.CodeAnalysis.CSharp.Scripting, 4.14.0"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Text.Json;
using System.Text.RegularExpressions;
using Microsoft.CodeAnalysis.CSharp.Scripting;
using Microsoft.CodeAnalysis.Scripting;

// Gains du PD canonique, strictement ordonnés.
int Gain(string mine, string other) => mine == "C" ? (other == "C" ? 3 : 0) : (other == "C" ? 5 : 1);
if (!(5 > 3 && 3 > 1 && 1 > 0 && 2 * 3 > 5 + 0))
    throw new Exception("PD non canonique");
foreach (var a in new[]
{
    "C",
    "D"
}

)
    foreach (var b in new[]
    {
        "C",
        "D"
    }

    )
        Console.WriteLine($"{a}/{b} -> {Gain(a, b)}");
Console.WriteLine("D domine strictement C dans ce jeu.");


Installed Packages Microsoft.CodeAnalysis.CSharp.Scripting, 4.14.0

C/C -> 3


C/D -> 0


D/C -> 5


D/D -> 1


D domine strictement C dans ce jeu.


## 2. Sémantique bornée et terminante du modèle

L'article définit les agents au moyen d'un `proof_search` (p. 7, §2.1) : un agent cherche une
**preuve formelle**, bornée en longueur par `k` caractères, d'un énoncé portant sur le code de
l'adversaire. Cette recherche présuppose un système de preuve (arithmétique de Peano ou une
extension) et une énumération des preuves ; elle n'est pas exécutable telle quelle dans un
notebook, et elle n'est pas décidable dans le cas non borné.

Ce notebook remplace la recherche de preuve par une **clôture finie et totale** :

1. chaque programme publie un **certificat** — un ensemble fini d'atomes tirés d'un vocabulaire
   fixe. C'est l'analogue des « bylaws » d'une institution, que la partie adverse peut lire ;
2. une liste finie de **règles d'inférence** dérive des faits sur l'adversaire à partir de son
   **seul** certificat ;
3. `proof_search(k, source, but)` teste si le but appartient à la clôture obtenue en **au plus
   `k` tours** d'application des règles ;
4. `k` compte des **tours d'inférence**, et non des caractères de preuve comme dans l'article :
   les seuils numériques ne sont donc **pas comparables**, seule la **monotonie** (`k` plus grand
   rend davantage de faits dérivables) est reproduite.

Trois limites de ce choix, à garder en tête pendant toute la lecture :

- la clôture est **finie** — l'univers des atomes dérivables est fixé à l'avance — donc la
  sémantique est **totale** et **terminante**, contrairement au `proof_search` non borné ;
- elle est **incomplète** vis-à-vis de l'article : elle ne peut pas dériver les faits qui
  exigent le lemme PBLT (lemme 3.6, p. 13), dont dépendent les théorèmes 3.4 et 3.7 ;
- un certificat **opaque** (vide) ne fournit aucun fait initial à la clôture. Dans le contrôle
  `DUPOC/CooperateBotOpake`, `DUPOC` fait défection : c'est l'analogue ciblé du point (3)
  de la p. 18. Ce n'est pas une propriété de tous les agents : `CooperateBot` et `CUPOD`
  coopèrent avec cet adversaire opaque dans la matrice finie.

In [2]:
// Neuf règles parallèles ; chaque tour lit seulement les faits du tour précédent.
int kDefaut = 3;
var rules = new (string premise, string[] conclusions)[]
{
    ("uncond_coop", new[] { "intent_coop" }),
    ("uncond_defect", new[] { "intent_defect" }),
    ("self_cond_coop", new[] { "implies_own_coop" }),
    ("intent_coop", new[] { "coops_against_any", "coops_with_db" }),
    ("intent_defect", new[] { "defects_against_any", "defects_with_db" }),
    ("needs_proof_of_coop", new[] { "defects_with_db" }),
    ("needs_proof_of_defect", new[] { "defects_with_db" }),
    ("self_cond_coop", new[] { "defects_with_db" }),
    ("prudent", new[] { "defects_with_db" })
};
Console.WriteLine($"Atomes certifiés : 6 ; règles : {rules.Length}");
foreach (var rule in rules)
    Console.WriteLine($"{rule.premise} => {string.Join(",", rule.conclusions)}");


Atomes certifiés : 6 ; règles : 9


uncond_coop => intent_coop


uncond_defect => intent_defect


self_cond_coop => implies_own_coop


intent_coop => coops_against_any,coops_with_db


intent_defect => defects_against_any,defects_with_db


needs_proof_of_coop => defects_with_db


needs_proof_of_defect => defects_with_db


self_cond_coop => defects_with_db


prudent => defects_with_db


### 2.1 Les sept programmes et leur certificat

Chaque programme est un texte C# : une fonction `nom(adverse, k, proofSearch)` et un champ
`bylaws_json` qui publie son certificat. Le certificat est **la** source d'information de
l'adversaire — un agent sans certificat est opaque.

| Programme | Rôle | Certificat publié |
| --- | --- | --- |
| `CooperateBot` | coopère toujours (p. 6) | `uncond_coop` |
| `DefectBot` | fait toujours défection (p. 6) | `uncond_defect` |
| `CooperateBotOpake` | coopère toujours, mais ne publie rien | aucun atome |
| `CUPOD` | fait défection s'il détient une preuve de défection (p. 9) | `needs_proof_of_defect` |
| `DUPOC` | coopère seulement sur preuve de coopération (p. 9 et 10) | `needs_proof_of_coop` |
| `CIMCIC` | coopère si sa coopération implique celle de l'autre (p. 20) | `self_cond_coop` |
| `PrudentBot_borne` | candidat borné de PrudentBot (Open Problem 9, p. 26) | `prudent`, `needs_proof_of_coop` |

Le certificat est publié **deux fois**, sous deux formes destinées à deux lecteurs distincts :
des marqueurs `// BYLAW:` lus par le moteur principal, et le champ `bylaws_json` lu par le
vérificateur indépendant. Cette redondance est délibérée : elle rend la comparaison des deux
implémentations significative (cf. section 8).

In [3]:
// Sept vraies fonctions C# sous forme de textes ; certificats lisibles deux fois.
string Source(string role, string body, string[] atoms)
{
    string markers = string.Join("\n", atoms.Select(x => "// BYLAW: " + x));
    string json = JsonSerializer.Serialize(new { atoms });
    string escaped = json.Replace("\\", "\\\\").Replace("\"", "\\\"");
    return $"string {role}(string adverse, int k, Func<int,string,string,bool> proofSearch)\n"
        + $"{{\n    {body}\n}}\n{markers}\nstring bylaws_json = \"{escaped}\";\n"
        + $"new Func<string,int,Func<int,string,string,bool>,string>({role})";
}

var sources = new Dictionary<string, string>
{
    ["CooperateBot"] = Source("CooperateBot", "return \"C\";", new[] { "uncond_coop" }),
    ["DefectBot"] = Source("DefectBot", "return \"D\";", new[] { "uncond_defect" }),
    ["CooperateBotOpake"] = Source("CooperateBotOpake", "return ((char)67).ToString();", Array.Empty<string>()),
    ["CUPOD"] = Source("CUPOD", "return proofSearch(k, adverse, \"defects_against_any\") ? \"D\" : \"C\";", new[] { "needs_proof_of_defect" }),
    ["DUPOC"] = Source("DUPOC", "return proofSearch(k, adverse, \"coops_against_any\") ? \"C\" : \"D\";", new[] { "needs_proof_of_coop" }),
    ["CIMCIC"] = Source("CIMCIC", "return proofSearch(k, adverse, \"implies_own_coop\") ? \"C\" : \"D\";", new[] { "self_cond_coop" }),
    ["PrudentBot_borne"] = Source("PrudentBot_borne", "return proofSearch(k, adverse, \"coops_against_any\") && "
        + "proofSearch(k, adverse, \"defects_with_db\") ? \"C\" : \"D\";", new[] { "prudent", "needs_proof_of_coop" })
};
var ordre = new[]
{
    "CooperateBot",
    "DefectBot",
    "CooperateBotOpake",
    "CUPOD",
    "DUPOC",
    "CIMCIC",
    "PrudentBot_borne"
};
foreach (var name in ordre)
    Console.WriteLine($"{name} : {sources[name].Split('\n')[0]}");
Console.WriteLine($"{sources.Count} textes de programmes et leurs certificats.");


CooperateBot : string CooperateBot(string adverse, int k, Func<int,string,string,bool> proofSearch)


DefectBot : string DefectBot(string adverse, int k, Func<int,string,string,bool> proofSearch)


CooperateBotOpake : string CooperateBotOpake(string adverse, int k, Func<int,string,string,bool> proofSearch)


CUPOD : string CUPOD(string adverse, int k, Func<int,string,string,bool> proofSearch)


DUPOC : string DUPOC(string adverse, int k, Func<int,string,string,bool> proofSearch)


CIMCIC : string CIMCIC(string adverse, int k, Func<int,string,string,bool> proofSearch)


PrudentBot_borne : string PrudentBot_borne(string adverse, int k, Func<int,string,string,bool> proofSearch)


7 textes de programmes et leurs certificats.


## 3. Moteur principal (implémentation 1)

Le moteur lit le certificat par ses marqueurs `// BYLAW:`, calcule la clôture bornée et expose
`proof_search(k, source, but)` aux programmes. La clôture s'arrête dès qu'un tour n'ajoute
plus rien : elle est donc terminante, et le résultat est **monotone** en `k`.

In [4]:
// Le moteur A lit uniquement les marqueurs et applique les neuf règles par tours.
HashSet<string> ParseA(string source) => Regex.Matches(source, @"(?m)^//\s*BYLAW:\s*([a-z_]+)\s*$").Select(m => m.Groups[1].Value).ToHashSet();
HashSet<string> ClosureA(IEnumerable<string> seed, int k)
{
    var facts = seed.ToHashSet();
    for (int t = 0; t < k; t++)
    {
        var next = new HashSet<string>(facts);
        foreach (var rule in rules)
            if (facts.Contains(rule.premise))
                next.UnionWith(rule.conclusions);
        if (next.SetEquals(facts))
            break;
        facts = next;
    }

    return facts;
}

bool ProofSearch(int k, string source, string goal) => ClosureA(ParseA(source), k).Contains(goal);
foreach (var name in new[]
{
    "CooperateBot",
    "DUPOC",
    "CIMCIC"
}

)
    Console.WriteLine($"{name} cert={string.Join(",", ParseA(sources[name]))} clôture={string.Join(",", ClosureA(ParseA(sources[name]), kDefaut).OrderBy(x => x))}");


CooperateBot cert=uncond_coop clôture=coops_against_any,coops_with_db,intent_coop,uncond_coop


DUPOC cert=needs_proof_of_coop clôture=defects_with_db,needs_proof_of_coop


CIMCIC cert=self_cond_coop clôture=defects_with_db,implies_own_coop,self_cond_coop


### 3.1 Exécution des programmes

La transparence est ici **effective** : chaque fonction reçoit le **texte** du programme adverse
et s'en sert pour interroger `proof_search`. On charge les sept programmes depuis leur texte,
en leur donnant accès au seul `proof_search` borné.

In [5]:
// Roslyn compile et exécute chaque texte : la décision vient de la fonction produite.
var options = ScriptOptions.Default.WithImports("System");
var programs = new Dictionary<string, Func<string, int, Func<int, string, string, bool>, string>>();
foreach (string name in ordre)
{
    programs[name] = await CSharpScript.EvaluateAsync<Func<string, int, Func<int, string, string, bool>, string>>(sources[name], options);
    Console.WriteLine($"{name} contre DefectBot -> {programs[name](sources["DefectBot"], kDefaut, ProofSearch)}");
}

Console.WriteLine($"{programs.Count} programmes compilés et appelés.");


CooperateBot contre DefectBot -> C


DefectBot contre DefectBot -> D


CooperateBotOpake contre DefectBot -> C


CUPOD contre DefectBot -> D


DUPOC contre DefectBot -> D


CIMCIC contre DefectBot -> D


PrudentBot_borne contre DefectBot -> D


7 programmes compilés et appelés.


### 3.2 Issue d'une confrontation en un coup

Une confrontation `(A, B)` fournit à chaque programme le texte de l'autre. L'issue est le
couple d'actions, suivi des deux gains.

In [6]:
// Deux exécutions de texte par confrontation ; 49 paires ordonnées.
(string a, string b, int ga, int gb) Issue(string a, string b, int k = 3)
{
    string x = programs[a](sources[b], k, ProofSearch);
    string y = programs[b](sources[a], k, ProofSearch);
    return (x, y, Gain(x, y), Gain(y, x));
}

var pairs = (
    from a in ordre
    from b in ordre
    select (a, b)).ToArray();
var outcomes = pairs.ToDictionary(p => p, p => Issue(p.a, p.b));
foreach (var a in ordre)
    Console.WriteLine($"{a, -18} " + string.Join(" ", ordre.Select(b => outcomes[(a, b)].a)));
Console.WriteLine("Gains du joueur ligne :");
foreach (var a in ordre)
    Console.WriteLine($"{a, -18} " + string.Join(" ", ordre.Select(b => outcomes[(a, b)].ga)));
Console.WriteLine($"{outcomes.Count} paires ordonnées.");


CooperateBot       C C C C C C C


DefectBot          D D D D D D D


CooperateBotOpake  C C C C C C C


CUPOD              C D C C C C C


DUPOC              C D D D D D D


CIMCIC             D D D D D C D


... (6 outputs left out)

DUPOC              3 1 5 5 1 1 1


CIMCIC             5 1 5 5 1 3 1


PrudentBot_borne   5 1 5 5 1 1 1


49 paires ordonnées.


### Lecture de la matrice

Quatre phénomènes sont visibles dans cette table (verdict **calculé** dans le modèle borné,
jamais une preuve) :

1. **Coopération mutuelle** : `DUPOC` contre `CooperateBot` donne `(C, C)` — le patient
   coopère parce que le certificat de `CooperateBot` est lisible, et réciproquement.
2. **Inexploitation** : `DUPOC` contre `CooperateBotOpake` donne `(D, C)`. Le contrôle opaque
   se comporte pourtant exactement comme `CooperateBot` ; seule sa **représentation** diffère.
3. **Défection révélée par la lecture** : `CUPOD` contre `DefectBot` donne `(D, D)` : `CUPOD`
   trouve dans le certificat de `DefectBot` la preuve qu'il va faire défection.
4. **Non-coopération sous transparence totale** : `DUPOC` contre `DUPOC` donne `(D, D)` — les
   deux programmes lisent tout le texte de l'autre et ne coopèrent pas pour autant. C'est le
   contrôle négatif de la section 5.

Les théorèmes 3.4 et 3.7 de l'article prévoient respectivement `(D, D)` pour
`CUPOD/CUPOD` et `(C, C)` pour `DUPOC/DUPOC` **quand `k` est grand** : l'écart entre ces énoncés et la table ci-dessus est
le sujet du classement de la section 4.

## 4. Classification : observation finie, résultat reproduit, théorème cité

Chaque affirmation de ce notebook appartient à **un seul** de ces trois régimes.

| Régime | Définition | Exemple dans ce notebook |
| --- | --- | --- |
| **Résultat reproduit** | Une affirmation de la source que le modèle borné retrouve, et qui **ne dépend pas** du lemme PBLT | Exemple 2.1, exemples 3.1 et 3.2, propositions 3.1 et 3.2, proposition 5.1, théorème 5.2(a) |
| **Observation finie** | Un calcul du modèle borné, **sans** énoncé correspondant dans la source, ou avec un énoncé qui en diffère | La table complète des 49 paires, la ligne de `PrudentBot_borne`, l'Open Problem 3 (laissé ouvert, section 9) |
| **Théorème cité** | Un énoncé de la source **cité** pour mémoire, que le modèle borné **ne reproduit pas** | Théorèmes 3.4 et 3.7 (PBLT), théorème 5.2(b) |

Le théorème 5.2(a) mérite une note : l'article le démontre **sans** invoquer PBLT pour l'essentiel,
car la preuve requise se réduit à l'implication `X => X` — une tautologie (p. 21). C'est pourquoi
le modèle borné le retrouve, alors qu'il échoue sur les théorèmes 3.4 et 3.7, dont la preuve passe
par le point fixe de PBLT.

In [7]:
// Le registre entier est contrôlé, sans prétendre démontrer les théorèmes cités.
var ex21 = Issue("CooperateBot", "DefectBot");
var bas = Issue("CUPOD", "DefectBot", 1);
var haut = Issue("CUPOD", "DefectBot", 3);
var ci = Issue("CIMCIC", "CIMCIC");
if ((ex21.a, ex21.b) != ("C", "D") || (bas.a, bas.b) != ("C", "D") || (haut.a, haut.b) != ("D", "D") || (ci.a, ci.b) != ("C", "C"))
    throw new Exception("Témoin divergent");
Console.WriteLine($"Exemple CB/DB : {ex21}; CUPOD/DB k=1 : {bas}, k=3 : {haut}; CIMCIC/CIMCIC : {ci}");
foreach (var(role, forbidden)in new[]
{
    ("CUPOD", "DC"),
    ("DUPOC", "CD"),
    ("CIMCIC", "CD")
}

)
{
    var witness = pairs.Where(p => p.a == role).FirstOrDefault(p => outcomes[p].a + outcomes[p].b == forbidden);
    if (witness.a != null)
        throw new Exception($"Témoin interdit {witness}");
    Console.WriteLine($"{role} : aucune issue {forbidden} sur le registre.");
}


Exemple CB/DB : (C, D, 0, 5); CUPOD/DB k=1 : (C, D, 0, 5), k=3 : (D, D, 1, 1); CIMCIC/CIMCIC : (C, C, 3, 3)


CUPOD : aucune issue DC sur le registre.


DUPOC : aucune issue CD sur le registre.


CIMCIC : aucune issue CD sur le registre.


## 5. Contrôle négatif : la transparence seule ne crée pas la coopération

Les programmes lisent ici **tout** le texte de l'adversaire. La question est de savoir si cette
transparence suffit à produire la coopération mutuelle. La réponse du modèle borné est non, et
c'est précisément l'endroit où il se sépare de l'article.

In [8]:
// Les deux contrôles négatifs restent des observations finies, sans PBLT.
var du = Issue("DUPOC", "DUPOC");
var cu = Issue("CUPOD", "CUPOD");
if ((du.a, du.b) != ("D", "D") || (cu.a, cu.b) != ("C", "C"))
    throw new Exception("Contrôle négatif divergent");
Console.WriteLine($"Observation finie DUPOC/DUPOC : {du} ; théorème 3.7 cité : (C,C).");
Console.WriteLine($"Observation finie CUPOD/CUPOD : {cu} ; théorème 3.4 cité : (D,D).");
Console.WriteLine($"Observation finie : outcome(DUPOC, DUPOC) = ('{du.a}', '{du.b}') ({du.ga}, {du.gb})");
Console.WriteLine($"Observation finie : outcome(CUPOD, CUPOD) = ('{cu.a}', '{cu.b}') ({cu.ga}, {cu.gb})");


Observation finie DUPOC/DUPOC : (D, D, 1, 1) ; théorème 3.7 cité : (C,C).


Observation finie CUPOD/CUPOD : (C, C, 3, 3) ; théorème 3.4 cité : (D,D).


Observation finie : outcome(DUPOC, DUPOC) = ('D', 'D') (1, 1)


Observation finie : outcome(CUPOD, CUPOD) = ('C', 'C') (3, 3)


## 6. Sortie non triviale : la représentation change une issue

`CooperateBot` et `CooperateBotOpake` ont un **comportement observable identique** : ils
coopèrent contre n'importe quel adversaire. Ils ne diffèrent que par le certificat publié.
Faites les confronter à `DUPOC`, et observez si la transparence du certificat change l'issue.

In [9]:
// Les deux textes coopèrent contre chaque adversaire ; leurs certificats diffèrent.
foreach (var opponent in ordre)
    if (programs["CooperateBot"](sources[opponent], 3, ProofSearch) != "C" || programs["CooperateBotOpake"](sources[opponent], 3, ProofSearch) != "C")
        throw new Exception("Comportements différents");
var readable = Issue("DUPOC", "CooperateBot");
var opaque = Issue("DUPOC", "CooperateBotOpake");
if ((readable.a, readable.b) != ("C", "C") || (opaque.a, opaque.b) != ("D", "C"))
    throw new Exception("Contrôle de représentation divergent");
Console.WriteLine($"DUPOC / lisible : {readable} ; DUPOC / opaque : {opaque}.");
Console.WriteLine($"DUPOC vs CooperateBot      : ('{readable.a}', '{readable.b}') ({readable.ga}, {readable.gb})");
Console.WriteLine($"DUPOC vs CooperateBotOpake : ('{opaque.a}', '{opaque.b}') ({opaque.ga}, {opaque.gb})");


DUPOC / lisible : (C, C, 3, 3) ; DUPOC / opaque : (D, C, 5, 0).


DUPOC vs CooperateBot      : ('C', 'C') (3, 3)


DUPOC vs CooperateBotOpake : ('D', 'C') (5, 0)


### Exercice 1 — une paire lisible / opaque, et la bascule qu'elle provoque

L'exemple guidé de la section 6 montre la bascule `(C, C)` vers `(D, C)` sur une paire fournie.
À vous de la reconstruire avec **vos propres** sources.

Objectif : écrire deux programmes de comportement identique (« coopérer toujours ») dont l'un
publie un certificat `uncond_coop` et l'autre rien, les charger, puis confronter `DUPOC` à
chacun et montrer que les issues diffèrent. Retourner un tuple contenant `lisible`, `opaque`
(les deux textes) et `issues` (les deux couples) pour permettre leur vérification.

Contraintes :

- les deux fonctions doivent porter **le même nom** `CooperateBot` : c'est une convention
  du contrat de chargement. `DUPOC` lit le certificat, pas ce nom ;
- le programme opaque ne doit pas contenir le littéral `"C"` dans un `return` : utilisez
  `(char)67` ;
- ne pas modifier `sources` ni `programs`.

Indices : `CSharpScript.EvaluateAsync` rend la fonction chargeable ; `Issue` exige un nom du registre : appelez directement le délégué `programs["DUPOC"]`
avec le texte test, `kDefaut` et `ProofSearch`.
Le délégué renvoie les deux textes et les deux couples ; le garde compile chaque texte.


In [10]:
// Exercice ouvert : fournir deux textes, leur compilation et les deux issues.
// Construire deux textes CooperateBot de même comportement et certificats différents.
// Compiler les textes, appeler DUPOC et renvoyer les deux issues sans modifier le registre.
Func<(string lisible, string opaque, (string, string)[] issues)?> exercice1 = () => null;
var reply1 = exercice1();
if (reply1 == null)
    Console.WriteLine("Exercice 1 à compléter.");
else
{
    var answer = reply1.Value;
    if (answer.lisible == null
        || answer.opaque == null
        || answer.lisible == answer.opaque
        || !Regex.IsMatch(answer.lisible, @"\bstring\s+CooperateBot\s*\(")
        || !Regex.IsMatch(answer.opaque, @"\bstring\s+CooperateBot\s*\(")
        || answer.opaque.Contains("return \"C\"")
        || !answer.opaque.Contains("(char)67")
        || !ParseA(answer.lisible).SetEquals(new[] { "uncond_coop" })
        || ParseA(answer.opaque).Count != 0)
        throw new Exception("Textes ou certificats incorrects");
    var f1 = await CSharpScript.EvaluateAsync<Func<string, int, Func<int, string, string, bool>, string>>(answer.lisible, options);
    var f2 = await CSharpScript.EvaluateAsync<Func<string, int, Func<int, string, string, bool>, string>>(answer.opaque, options);
    foreach (var opponent in ordre)
        if (f1(sources[opponent], 3, ProofSearch) != "C" || f2(sources[opponent], 3, ProofSearch) != "C")
            throw new Exception("Comportements différents");
    var expected = new[]
    {
        (programs["DUPOC"](answer.lisible, 3, ProofSearch), f1(sources["DUPOC"], 3, ProofSearch)),
        (programs["DUPOC"](answer.opaque, 3, ProofSearch), f2(sources["DUPOC"], 3, ProofSearch))
    };
    if (answer.issues == null || !answer.issues.SequenceEqual(expected) || !expected.SequenceEqual(new[] { ("C", "C"), ("D", "C") }))
        throw new Exception("Issues incorrectes");
    Console.WriteLine("Exercice 1 : sondes réussies.");
}


Exercice 1 à compléter.


## 7. Rôle de la borne `k`

La borne `k` est le seul paramètre libre du modèle. Un tour d'inférence ne peut produire un fait
que si ses prémisses sont déjà présentes : les bascules se produisent donc à des seuils précis.
L'exemple guidé ci-dessous balaie `k` de 0 à 4 sur quatre confrontations.

In [11]:
// Quatre paires et cinq budgets, k comptant des tours plutôt qu'une longueur de preuve.
for (int k = 0; k <= 4; k++)
{
    var values = new[]
    {
        Issue("CUPOD", "DefectBot", k),
        Issue("DUPOC", "CooperateBot", k),
        Issue("CIMCIC", "CIMCIC", k),
        Issue("DUPOC", "DUPOC", k)
    };
    Console.WriteLine($"k={k} : " + string.Join(" | ", values.Select(v => $"({v.a},{v.b})")));
}

if ((Issue("CUPOD", "DefectBot", 1).a, Issue("CUPOD", "DefectBot", 1).b) != ("C", "D") || (Issue("CUPOD", "DefectBot", 2).a, Issue("CUPOD", "DefectBot", 2).b) != ("D", "D"))
    throw new Exception("Seuil incorrect");
Console.WriteLine("CUPOD / DefectBot bascule à k=2 ; la clôture est monotone et se stabilise.");
Console.WriteLine("La cloture est monotone et se stabilise : k plus grand n'ajoute plus aucun fait.");


k=0 : (C,D) | (D,C) | (D,D) | (D,D)


k=1 : (C,D) | (D,C) | (C,C) | (D,D)


k=2 : (D,D) | (C,C) | (C,C) | (D,D)


k=3 : (D,D) | (C,C) | (C,C) | (D,D)


k=4 : (D,D) | (C,C) | (C,C) | (D,D)


CUPOD / DefectBot bascule à k=2 ; la clôture est monotone et se stabilise.


La cloture est monotone et se stabilise : k plus grand n'ajoute plus aucun fait.


### Exercice 2 — borne de stabilisation sur toutes les paires

L'exemple guidé balaie quatre confrontations choisies. À vous de traiter le registre entier.

Objectif : calculer une borne de stabilisation pour une paire et `kMax=5` qui rend la **plus petite**
borne `k` telle que l'issue ne change plus pour tout `k` jusqu'à `k_max`, puis l'appliquer aux
49 paires ordonnées et afficher la table.

Contraintes : utiliser `Issue` et `ordre`, puis parcourir les paires avec LINQ ou des boucles ;
ne pas recopier la table déjà affichée.

Indices : « l'issue ne change plus » signifie que toutes les issues de `k` à `k_max` sont égales à celle en `k_max`.
Cas limite à signaler explicitement : une paire peut déjà être stable en `k = 0`.
En C#, remplir le délégué paramétré par `kMax`.


In [12]:
// Exercice ouvert : stabilisation minimale sur toutes les 49 paires.
// Pour chaque paire, chercher la première borne dont tout le suffixe jusqu’à kMax est stable.
// Parcourir les 49 paires et ne pas recopier la table du moteur.
Func<int, Dictionary<(string, string), int>> exercice2 = kMax => null;
Dictionary<(string, string), int> Oracle2(int max)
{
    var table = new Dictionary<(string, string), int>();
    foreach (var p in pairs)
    {
        var values = Enumerable.Range(0, max + 1).Select(k => Issue(p.a, p.b, k)).ToArray();
        table[p] = Enumerable.Range(0, max + 1).First(k => values.Skip(k).All(v => v == values[max]));
    }

    return table;
}

var limits = new[]
{
    0,
    2,
    5
};
var expected2 = limits.Select(Oracle2).ToArray();
var replies2 = limits.Select(exercice2).ToArray();
if (replies2.All(r => r == null))
    Console.WriteLine("Exercice 2 à compléter.");
else
{
    if (replies2.Any(r => r == null))
        throw new Exception("Réponse partielle");
    for (int i = 0; i < limits.Length; i++)
        if (replies2[i].Count != expected2[i].Count || expected2[i].Any(p => !replies2[i].ContainsKey(p.Key) || replies2[i][p.Key] != p.Value))
            throw new Exception("Table incorrecte");
    Console.WriteLine("Exercice 2 : sondes réussies.");
}


Exercice 2 à compléter.


## 8. Vérificateur indépendant (implémentation 2)

Un résultat calculé par un seul programme n'est pas vérifié. On recalcule toute la table par un
**second chemin de calcul**, écrit indépendamment sur trois plans :

| Plan | Moteur principal | Vérificateur |
| --- | --- | --- |
| Lecture du certificat | marqueurs `// BYLAW:` (expression régulière) | champ `bylaws_json` (`JsonSerializer` et `JsonDocument`) |
| Clôture | parcours d'une liste de couples d'ensembles, avec arrêt anticipé | itération naïve jusqu'au point fixe sur un dictionnaire prémisses vers conclusions |
| Décision | **exécution** de la fonction programme | **réécriture** de la règle de décision par rôle, sans appel au moteur |

La **spécification** — atomes, règles, table de gains — est commune : c'est elle que la
comparaison met à l'épreuve. Ce qui est indépendant, c'est le code qui l'implémente.

In [13]:
// Moteur B : parsing JSON, règles regroupées, décisions réécrites par rôle.
var rulesB = new Dictionary<string, string[]>
{
    ["uncond_coop"] = new[]
    {
        "intent_coop"
    },
    ["uncond_defect"] = new[]
    {
        "intent_defect"
    },
    ["self_cond_coop"] = new[]
    {
        "implies_own_coop",
        "defects_with_db"
    },
    ["intent_coop"] = new[]
    {
        "coops_against_any",
        "coops_with_db"
    },
    ["intent_defect"] = new[]
    {
        "defects_against_any",
        "defects_with_db"
    },
    ["needs_proof_of_coop"] = new[]
    {
        "defects_with_db"
    },
    ["needs_proof_of_defect"] = new[]
    {
        "defects_with_db"
    },
    ["prudent"] = new[]
    {
        "defects_with_db"
    }
};
HashSet<string> ParseB(string source)
{
    var match = Regex.Match(source, "bylaws_json\\s*=\\s*\"((?:\\\\.|[^\"])*)\"");
    if (!match.Success)
        throw new Exception("JSON absent");
    string payload = JsonSerializer.Deserialize<string>("\"" + match.Groups[1].Value + "\"");
    using (var doc = JsonDocument.Parse(payload))
        return doc.RootElement.GetProperty("atoms").EnumerateArray().Select(x => x.GetString()).ToHashSet();
}

HashSet<string> ClosureB(IEnumerable<string> seed, int k)
{
    var facts = seed.ToHashSet();
    for (int t = 0; t < k; t++)
    {
        var additions = new HashSet<string>();
        foreach (var rule in rulesB)
            if (facts.Contains(rule.Key))
                additions.UnionWith(rule.Value);
        if (additions.IsSubsetOf(facts))
            break;
        facts.UnionWith(additions);
    }

    return facts;
}

string DecideB(string role, string source, int k)
{
    var facts = ClosureB(ParseB(source), k);
    return role switch
    {
        "CooperateBot" or "CooperateBotOpake" => "C",
        "DefectBot" => "D",
        "CUPOD" => facts.Contains("defects_against_any") ? "D" : "C",
        "DUPOC" => facts.Contains("coops_against_any") ? "C" : "D",
        "CIMCIC" => facts.Contains("implies_own_coop") ? "C" : "D",
        "PrudentBot_borne" => facts.Contains("coops_against_any") && facts.Contains("defects_with_db") ? "C" : "D",
        _ => throw new ArgumentException(role)};
}

(string a, string b, int ga, int gb) IssueB(string a, string b, int k = 3)
{
    var x = DecideB(a, sources[b], k);
    var y = DecideB(b, sources[a], k);
    return (x, y, Gain(x, y), Gain(y, x));
}

foreach (var name in ordre)
    if (!ParseA(sources[name]).SetEquals(ParseB(sources[name])))
        throw new Exception("Certificats divergents");
Console.WriteLine($"Deux lectures de certificats concordent pour {ordre.Length} programmes ; {rulesB.Count} prémisses B.");
Console.WriteLine($"Regles du verificateur : {rulesB.Count} premisses, ecrites independamment.");


Deux lectures de certificats concordent pour 7 programmes ; 8 prémisses B.


Regles du verificateur : 8 premisses, ecrites independamment.


### 8.1 Comparaison mécanique des deux implémentations

On compare non seulement les actions, mais la **relation de coopération** et la **matrice de
gains**. Un désaccord sur n'importe laquelle des trois serait un diagnostic.

In [14]:
// Les actions, la relation de coopération et les gains sont comparés sur 49 paires.
var mismatches = pairs.Where(p => outcomes[p] != IssueB(p.a, p.b)).ToArray();
var relationA = pairs.Where(p => outcomes[p].a == "C").ToHashSet();
var relationB = pairs.Where(p => IssueB(p.a, p.b).a == "C").ToHashSet();
bool gainsEqual = pairs.All(p => (outcomes[p].ga, outcomes[p].gb) == (IssueB(p.a, p.b).ga, IssueB(p.a, p.b).gb));
if (mismatches.Length != 0 || !relationA.SetEquals(relationB) || !gainsEqual)
    throw new Exception("Désaccord entre moteurs");
Console.WriteLine($"Paires ordonnées comparées : {pairs.Length}, désaccords : {mismatches.Length}");
Console.WriteLine($"Relation de coopération identique : {relationA.SetEquals(relationB)} ; matrice de gains identique : {gainsEqual}");
Console.WriteLine($"Paires ordonnees comparees : {pairs.Length}");
Console.WriteLine($"Desaccords moteur / verificateur : {mismatches.Length}");
Console.WriteLine($"Relation de cooperation identique : {relationA.SetEquals(relationB)}");
Console.WriteLine($"Matrice de gains identique        : {gainsEqual}");


Paires ordonnées comparées : 49, désaccords : 0


Relation de coopération identique : True ; matrice de gains identique : True


Paires ordonnees comparees : 49


Desaccords moteur / verificateur : 0


Relation de cooperation identique : True


Matrice de gains identique        : True


### Exercice 3 — une troisième vérification des paires

Le vérificateur de la section 8 est un second chemin. Écrivez un **troisième** contrôle
qui reconstruit et agrège les paires à partir de sa fonction de décision, puis retrouve la
non-exploitabilité de `CUPOD`, `DUPOC` et `CIMCIC`
(propositions 3.1, 3.2 et 5.1) sur **toutes** les paires du registre. Cette reconstruction
est indépendante de la table déjà calculée, mais partage la décision du moteur B : ce n'est
pas une troisième implémentation indépendante de la règle de décision.

Objectif : rendre un dictionnaire de témoins `{role: paire_temoin ou null}`, où une paire
témoin est une paire dont le couple d'actions est l'issue interdite du rôle
(`CUPOD` interdit `DC`, `DUPOC` interdit `CD`, `CIMCIC` interdit `CD`).

Contraintes : ne pas utiliser `outcomes` ni `IssueB`. Partir du délégué de décision reçu
(`DecideB` pour le banc) et de `Gain` uniquement.

Indice : le délégué de décision rend une action isolée ; appelez-le deux fois pour reconstruire
un couple d'actions. Le contrôle réussi rend trois valeurs `null`.
En C#, remplir le délégué qui reçoit une fonction de décision ; une sonde synthétique teste aussi la détection de témoins.


Les orientations diffèrent selon le rôle : pour CUPOD, le témoin interdit D/C vérifierait qu’il profite de la coopération adverse ; pour DUPOC et CIMCIC, C/D signifierait qu’ils subissent la défection adverse. Le contrôle n’assimile donc pas ces deux sens de non-exploitation.


In [15]:
// Exercice ouvert : un troisième contrôle doit aussi trouver un témoin synthétique.
// Reconstituer les actions avec la fonction de décision passée en argument.
// Pour chacun des trois rôles, renvoyer un témoin interdit ou null.
Func<Func<string, string, int, string>, Dictionary<string, (string, string)?>> exercice3 = decider => null;
Func<string, string, int, string> badDecider = (name, source, k) => name == "CUPOD" ? "D" : "C";
var replies3 = new[]
{
    exercice3(DecideB),
    exercice3(badDecider)
};
if (replies3.All(r => r == null))
    Console.WriteLine("Exercice 3 à compléter.");
else
{
    if (replies3.Any(r => r == null))
        throw new Exception("Réponse partielle");
    var expected = new Dictionary<string, (string, string)?>
    {
        ["CUPOD"] = ("CUPOD", ordre[0]),
        ["DUPOC"] = ("DUPOC", "CUPOD"),
        ["CIMCIC"] = ("CIMCIC", "CUPOD")
    };
    if (replies3[0].Count != 3
        || expected.Keys.Any(k => !replies3[0].ContainsKey(k)
        || replies3[0][k] != null)
        || replies3[1].Count != 3
        || expected.Any(k => !replies3[1].ContainsKey(k.Key)
        || replies3[1][k.Key] != k.Value))
        throw new Exception("Témoins incorrects");
    Console.WriteLine("Exercice 3 : sondes réussies.");
}


Exercice 3 à compléter.


## 9. Open Problem 3 : DUPOC(k) contre CUPOD(k)

L'Open Problem 3 (p. 18) conjecture, pour `k` grand,
`outcome(DUPOC(k), CUPOD(k)) == (D, C)`, et demande si c'est bien le cas. L'article explique
pourquoi la question est difficile : l'argument de symétrie naïf est invalide, car `DUPOC(k)`
n'est pas le miroir exact de `CUPOD(k)` — les lettres `C` et `D` ne sont pas échangées dans le
`proof_checker`. Il faut raisonner sur l'échec d'une recherche de preuve d'un agent pendant que
l'autre ne peut pas prouver cet échec, ce que PBLT ne permet pas.

**Cet Open Problem reste explicitement ouvert.** Le modèle borné calcule une issue pour cette
paire, comme pour toutes les autres, mais ce calcul **ne tranche pas** la conjecture : il
n'implémente pas la notion de preuve non bornée sur laquelle porte l'énoncé. La cellule qui suit
affiche l'observation et le dit ; aucun exercice du notebook ne porte sur ce problème.

In [16]:
// L'observation finie ne tranche pas l'Open Problem 3.
Console.WriteLine($"Observation bornée DUPOC/CUPOD : {Issue("DUPOC", "CUPOD")}");
Console.WriteLine($"Observation bornée CUPOD/DUPOC : {Issue("CUPOD", "DUPOC")}");
Console.WriteLine("Open Problem 3 reste ouvert : cette clôture ne démontre aucun résultat PBLT.");


Observation bornée DUPOC/CUPOD : (D, C, 5, 0)


Observation bornée CUPOD/DUPOC : (C, D, 0, 5)


Open Problem 3 reste ouvert : cette clôture ne démontre aucun résultat PBLT.


## 10. Les dix problèmes ouverts de la source

| # | Énoncé | Statut dans la source | Prérequis | Expérience finie possible |
| --- | --- | --- | --- | --- |
| 1 | Prouver le théorème de Löb sans le point fixe modal `Psi <-> (Box Psi -> C)` | Open Problem 1, p. 15, théorie de la preuve | Logique de la prouvabilité, théorème du point fixe modal | Non : la question est l'existence d'une preuve, non un calcul. Au mieux, tester une formalisation partielle. |
| 2 | Conditions sous lesquelles une population contenant des DUPOC évolue vers des agents G-fair | Open Problem 2, p. 18, dynamique de populations | Dynamique évolutionnaire, simulations | Oui : tournoi répété sur une population finie de petits certificats, mesurer la part de G-fair. |
| 3 | Pour `k` grand, `outcome(DUPOC(k), CUPOD(k)) == (D, C)` | Open Problem 3, p. 18 — **problème explicitement ouvert** | Raisonner sur l'échec d'une recherche de preuve non bornée ; PBLT ne s'applique pas | **Non résolutoire** : le modèle borné calcule une issue, mais celle-ci ne tranche pas l'énoncé. Aucun exercice ne porte sur ce problème. |
| 4 | Implémenter DUPOC par recherche heuristique de preuve en HOL/ML ou Coq, et vérifier l'arrêt coopératif sur une machine de bureau | Open Problem 4, p. 20, ingénierie de la preuve | HOL/ML ou Coq, tactiques heuristiques | Oui, mais hors notebook : exige un assistant de preuve réel. |
| 5 | Que vaut `outcome(CUPOD(k), CIMCIC(k))` | Open Problem 5, p. 22 | Même difficulté que le problème 3 | Non résolutoire : observation bornée possible, sans valeur de preuve. |
| 6 | Que vaut `outcome(DUPOC(k), DIMCID(k))` | Open Problem 6, p. 24 | Définition de DIMCID, p. 22 | Non résolutoire ; DIMCID n'est pas défini dans ce notebook. |
| 7 | Généraliser PBLT en analogue borné de Gödel-Löb suivant les longueurs de preuve | Open Problem 7, p. 25 et 26 | Logique modale, PBLT, Critch 2016 théorème 4.2 | Partielle : instrumenter les longueurs dans un système de preuve jouet. |
| 8 | Appliquer le résultat du problème 7 pour améliorer l'algorithme Haskell du dépôt publique `klao/provability` | Open Problem 8, p. 26 | Haskell, sémantique de Kripke | Oui, mais dépend du problème 7. |
| 9 | Existe-t-il une version bornée de PrudentBot | Open Problem 9, p. 26 | Définition de PrudentBot (LaVictoire et al. 2017, cité p. 26) | Oui : le candidat `PrudentBot_borne` de ce notebook en est un essai. Il ne coopère avec aucun agent du registre, ce qui **illustre** la difficulté sans résoudre le problème 9. |
| 10 | Implémenter un CDEBot borné obtenant `(C, C)` contre DUPOC(k) et `(D, D)` contre EUPOD(k) | Open Problem 10, p. 27 | PD étendu à trois actions (E), application conditionnelle de PBLT | Oui : matrice bornée sur le jeu étendu à trois actions. |

Les dix énoncés ci-dessus sont ceux de la source, avec leur numérotation et leur page. Les
problèmes 3, 5 et 6 sont ceux que l'article désigne lui-même comme ne relevant pas d'une
« self-fulfilling prophecy » démontrable par PBLT (p. 32 et 33).

## 11. Limites et résidu

- **Le modèle n'est pas la source.** La clôture finie n'est pas le `proof_search` de l'article :
  aucun énoncé impliquant PBLT (théorèmes 3.4, 3.7, 5.2(b)) n'est reproduit. Ces énoncés sont
  cités comme tels dans le tableau de la section 4 et dans le contrôle négatif.
- **Les seuils de `k` ne sont pas transposables.** L'article compte des caractères de preuve, ce
  notebook des tours d'inférence. Seule la monotonie est commune.
- **`PrudentBot_borne` est un candidat, pas une solution.** Sa ligne dans la matrice est
  entièrement `D` : au sens du modèle, il ne coopère avec aucun agent du registre. C'est
  cohérent avec le fait que le PrudentBot de LaVictoire et al. repose sur une **recherche de
  preuve supplémentaire** dont l'analogue borné est précisément l'Open Problem 9, laissé ouvert.
- **Les certificats sont déclaratifs.** Un programme peut mentir sur son certificat ; le modèle
  suppose la publication sincère, comme l'article suppose un `proof_check` correct. La question
  de la vérification des certificats n'est pas traitée ici.
- **`CooperateBotOpake` est un contrôle local**, sans équivalent publié : il sert uniquement à
  isoler l'effet de la représentation dans la section 6.

## 12. Conclusion

Ce notebook a défini une sémantique **bornée et terminante** des agents `CUPOD`, `DUPOC`,
`PrudentBot` et `CIMCIC`, puis a classé chaque affirmation dans un des trois régimes
— **observation finie**, **résultat reproduit**, **théorème cité**.

À retenir :

1. Dans le jeu de programmes transparents, la lecture des certificats change l'issue observée : `DUPOC` obtient
   `(C, C)` contre `CooperateBot`, et `(D, D)` contre `DefectBot`.
2. La transparence **seule** ne suffit pas : `DUPOC` contre `DUPOC` donne `(D, D)` dans le
   modèle borné, alors que le théorème 3.7 donne `(C, C)` via PBLT. Le lemme manquant est
   exactement ce que le modèle ne peut pas reproduire.
3. La **représentation** change l'issue à comportement constant : `CooperateBot` et
   `CooperateBotOpake` coopèrent tous deux contre tous, mais `DUPOC` ne récompense que le
   premier — le point (3) de la p. 18.
4. Les propositions 3.1, 3.2 et 5.1 sont reproduites sur tout le registre, et le théorème 5.2(a)
   l'est aussi, parce que sa preuve se réduit à une tautologie.
5. Vérifier demande **deux chemins** : le moteur et le vérificateur sont indépendants dans leur
   lecture du certificat, leur algorithme de clôture et leur évaluateur de décision, et
   s'accordent sur les actions, la relation de coopération et la matrice de gains.
6. **Open Problem 3 reste ouvert** : le notebook l'observe sans le trancher, et n'en fait ni un
   exercice ni un résultat.

Pour aller plus loin : les dix problèmes ouverts de la section 10, et les notebooks voisins
`GameTheory-06e` (cadre des jeux-programmes) et `GameTheory-06g` (analogue par simulation).

## À retenir

- **La transparence seule ne suffit pas** : `Observation finie : outcome(DUPOC, DUPOC) = ('D', 'D') (1, 1)` — le théorème 3.7 donne `(C, C)` pour `k` grand via PBLT : sans le lemme de prouvabilité, les deux font défection.
- **L'écart est en miroir** : `Observation finie : outcome(CUPOD, CUPOD) = ('C', 'C') (3, 3)` là où le théorème 3.4 donne `(D, D)` — le modèle borné est incomplet dans les deux sens, et le carnet le dit.
- **La représentation décide du mérite** : `DUPOC vs CooperateBot      : ('C', 'C') (3, 3)` et `DUPOC vs CooperateBotOpake : ('D', 'C') (5, 0)` — deux adversaires au comportement identique, l'un au certificat publié et l'autre au certificat vide : l'issue change.
- **Moteur et vérificateur s'accordent** : `Desaccords moteur / verificateur : 0` sur `Paires ordonnees comparees : 49`, avec `Relation de cooperation identique : True` et `Matrice de gains identique        : True`.
- **La clôture est monotone** : `La cloture est monotone et se stabilise : k plus grand n'ajoute plus aucun fait.` — et `CUPOD contre DefectBot bascule de (C, D) a (D, D) des que k atteint 2 tours.`
- **Un problème ouvert reste ouvert** : `Open Problem 3 (p. 18)` — le carnet l'observe sans le trancher, et aucun exercice ne porte dessus.


## Vérifiez votre compréhension

1. Que montrent `Observation finie : outcome(DUPOC, DUPOC) = ('D', 'D') (1, 1)` et le théorème 3.7 cité, et que doit-on en conclure ?

   <details><summary>Réponse</summary>

   **Réponse.** Le modèle borné produit `('D', 'D')` pour `DUPOC` contre `DUPOC`, là où le théorème 3.7 (p. 14) annonce `outcome(DUPOC(k), DUPOC(k)) == (C, C)` pour `k` grand, via le lemme 3.6 (PBLT). La conclusion honnête est double : le modèle **ne reproduit pas** le théorème, il en mesure l'écart — la transparence mutuelle ne suffit pas sans le raisonnement de prouvabilité, que la borne ne sait pas représenter. L'observation est un **résultat du modèle**, pas un résultat de la source.

   </details>

2. Que prouve `Desaccords moteur / verificateur : 0` sur `Paires ordonnees comparees : 49` — et que ne prouve pas ?

   <details><summary>Réponse</summary>

   **Réponse.** Que les deux implémentations indépendantes — le moteur, qui lit les marqueurs `// BYLAW:`, et le vérificateur, dont `Regles du verificateur : 8 premisses, ecrites independamment.` — produisent le même verdict sur 49 paires ordonnées : même relation de coopération, même matrice de gains. Cela **contrôle l'implémentation**, pas la sémantique : les deux peuvent se tromper ensemble sur la question de la bornitude, et le carnet le dit dans sa section des limites.

   </details>


## Pour aller plus loin

**Carnet suivant.** [Dynamics-09 — Biens publics et N personnes](Dynamics-09-Biens-Publics-N-Personnes-CSharp.ipynb) — le passage de deux agents à N joueurs : ce que la transparence des programmes dit du bien public.

**Carnets voisins.**

- [Dynamics-04 — Open-source game theory](Dynamics-04-Open-Source-Game-Theory-Python.ipynb) — le moteur dont ce carnet reprend l'instrument de simulation.
- [Dynamics-04b — FairBot et Loeb (Lean)](Dynamics-04b-FairBot-Loeb-Lean.ipynb) — la version formelle d'un agent qui lit la source de l'adversaire.
- [Dynamics-05 — Agents bornés (Python)](Dynamics-05-Agents-Bornes-Python.ipynb) — la famille de budgets et le miroir décidable.

**Lectures.**

- Critch, Dennis & Russell (2022), *Cooperative and uncooperative institution designs: Surprises and problems in open-source game theory*, arXiv:2208.07006 — notice héritée (théorèmes 3.4 et 3.7, lemme 3.6, problèmes ouverts).
- LaVictoire et al. (2017) — `PrudentBot`, l'agent non borné dont ce carnet implémente une version bornée (p. 26 de la source).
